# 04 - Modelos Avanzados: SVD, SVM, Random Forest y XGBoost
## Clasificación del Grado de Artrosis de Rodilla mediante Machine Learning

**Asignatura:** Machine Learning 1 (23433)
**Carrera:** Ingeniería Electrónica - FIUNA
**Metodología:** Basada en el repositorio de la cátedra ([carlobeni/Machine-Learning](https://github.com/carlobeni/Machine-Learning)), Clase 8 (SVD/Subespacios) y TP4 (Benchmark Final).

---

### Objetivos del Notebook:
1. **Clasificador de Subespacios SVD (Teorema de Eckart-Young):** Generalización multiclase del método de menor norma residual de la Clase 8, proyectando cada radiografía aplanada ($64 \times 64 = 4096$ píxeles) sobre los $K=5$ subespacios truncados de los grados Kellgren-Lawrence.
2. **Support Vector Machine (SVM) con Kernel Gaussiano RBF:** Proyección implícita al espacio de Hilbert de dimensión infinita mediante el *kernel trick* para resolver fronteras no lineales entre los 320 descriptores tabulares.
3. **Random Forest:** Ensamble de árboles con ajuste de hiperparámetros seleccionado por F1 macro en validación.
4. **XGBoost (Extreme Gradient Boosting):** Boosting de árboles para modelar interacciones no lineales entre los descriptores HOG, Sobel, Laplaciano e intensidad.
5. **Benchmark comparativo:** Métricas, matrices de confusión, curvas ROC multiclase y curvas de aprendizaje junto con el baseline del Notebook 03.



In [1]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import StratifiedKFold, learning_curve
from xgboost import XGBClassifier

plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

PROJECT_DIR = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "processed").exists()), None)

if PROJECT_DIR is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR    = PROJECT_DIR / "models"
REPORTS_DIR   = PROJECT_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

class_names  = ["0 - Normal", "1 - Dudosa", "2 - Leve", "3 - Moderada", "4 - Severa"]
classes_short = ["Normal", "Dudosa", "Leve", "Moderada", "Severa"]

print(f"Directorio del proyecto: {PROJECT_DIR}")


Directorio del proyecto: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla


## 1. Carga de Datos Preprocesados

Cargamos las dos representaciones complementarias generadas en `02_Preprocesamiento.ipynb`:
- **Características tabulares** ($D=322$): HOG, gradientes, estadísticos de intensidad, pirámides espaciales y geometría. Utilizadas por SVM, XGBoost y Regresión Logística.
- **Vectores planos** ($D=4096$): Radiografía acondicionada a $64 \times 64$ píxeles aplanada. Utilizados exclusivamente por el Clasificador de Subespacios SVD.

Se concatenan las particiones de entrenamiento y validación ($\mathcal{D}_{\text{train+val}}$, $N=1402$) para el entrenamiento final de todos los modelos. El conjunto de test ($N=248$) se mantiene completamente aislado.

In [2]:
data_arrays = np.load(PROCESSED_DIR / "features_arrays.npz")

X_train     = data_arrays["X_train_tabular"]
y_train     = data_arrays["y_train"]
X_val       = data_arrays["X_val_tabular"]
y_val       = data_arrays["y_val"]
X_test      = data_arrays["X_test_tabular"]
y_test      = data_arrays["y_test"]
feature_names = data_arrays["feature_names"]

X_train_flat = data_arrays["X_train_flat"]
X_val_flat   = data_arrays["X_val_flat"]
X_test_flat  = data_arrays["X_test_flat"]

# Concatenar train + val para entrenamiento final
X_train_final      = np.vstack([X_train, X_val])
y_train_final      = np.concatenate([y_train, y_val])
X_train_flat_final = np.vstack([X_train_flat, X_val_flat])

print(f"Características tabulares: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")
print(f"Vectores planos (64x64):   Train={X_train_flat.shape}, Val={X_val_flat.shape}, Test={X_test_flat.shape}")
print(f"Desarrollo (Train+Val):    Tabular={X_train_final.shape}, Flat={X_train_flat_final.shape}")
print(f"Test independiente:        N={X_test.shape[0]}")

Características tabulares: Train=(1155, 320), Val=(247, 320), Test=(248, 320)
Vectores planos (64x64):   Train=(1155, 4096), Val=(247, 4096), Test=(248, 4096)
Desarrollo (Train+Val):    Tabular=(1402, 320), Flat=(1402, 4096)
Test independiente:        N=248


## 2. Clasificador de Subespacios SVD (Metodología de Clase 8)

### Fundamento Teórico

Según el **Teorema de Eckart-Young-Mirsky**, la mejor aproximación de rango $r$ de una matriz $\mathbf{A}$ en norma de Frobenius es la SVD truncada:

$$\mathbf{A}_r = \sum_{i=1}^{r} \sigma_i \, \mathbf{u}_i \, \mathbf{v}_i^T$$

Para clasificación, construimos un subespacio $\mathcal{S}_k$ para cada grado Kellgren-Lawrence $k \in \{0,1,2,3,4\}$:
1. **Centrado:** $\tilde{\mathbf{X}}_k = \mathbf{X}_k - \boldsymbol{\mu}_k$
2. **Descomposición SVD:** $\tilde{\mathbf{X}}_k^T = \mathbf{U}_k \, \boldsymbol{\Sigma}_k \, \mathbf{V}_k^T$
3. **Truncamiento:** Retener las $r$ primeras columnas de $\mathbf{U}_k$ (las $r$ direcciones de mayor varianza).

Para clasificar una nueva muestra $\mathbf{z}$, se calcula la **norma del residuo de proyección** sobre cada subespacio:

$$\hat{y} = \arg\min_{k} \left\| (\mathbf{z} - \boldsymbol{\mu}_k) - \mathbf{U}_k^{(r)} \, {\mathbf{U}_k^{(r)}}^T \, (\mathbf{z} - \boldsymbol{\mu}_k) \right\|_2$$

La clase asignada es aquella cuyo subespacio produce el **menor error de reconstrucción** (menor norma residual).

### Generalización Multiclase

El clasificador de la cátedra está implementado para 2 clases (gatos/perros). Aquí lo generalizamos a $K=5$ subespacios independientes, uno por cada grado de artrosis.

In [3]:
class SVDSubspaceClassifier(BaseEstimator, ClassifierMixin):
    """
    Clasificador de Subespacios SVD Multiclase (Subspace Method / Menor Norma Residual).

    Generalización del clasificador binario de la cátedra a K clases.
    Para cada clase k, construye un subespacio truncado U_k de rango r
    y clasifica según la menor norma residual de proyección.

    Parámetros:
    -----------
    k_components : int
        Número de componentes singulares a retener por clase (rango r).
    temperature : float
        Temperatura del softmax para calibrar probabilidades.
    """
    def __init__(self, k_components=20, temperature=0.25):
        self.k_components = k_components
        self.temperature = temperature

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.means_ = {}
        self.bases_ = {}
        self.k_ = {}

        for cls in self.classes_:
            X_cls = X[y == cls]
            mu = np.mean(X_cls, axis=0)
            self.means_[cls] = mu
            X_centered = X_cls - mu
            U, S, _ = np.linalg.svd(X_centered.T, full_matrices=False)
            k = min(self.k_components, U.shape[1])
            self.bases_[cls] = U[:, :k]
            self.k_[cls] = k

        return self

    def _compute_residuals(self, X):
        X = np.asarray(X, dtype=np.float64)
        n = X.shape[0]
        norms = np.zeros((n, len(self.classes_)))
        for j, cls in enumerate(self.classes_):
            U = self.bases_[cls]
            mu = self.means_[cls]
            diff = X - mu
            proj = diff @ U @ U.T
            residual = diff - proj
            norms[:, j] = np.linalg.norm(residual, axis=1)
        return norms

    def predict(self, X):
        norms = self._compute_residuals(X)
        return self.classes_[np.argmin(norms, axis=1)]

    def predict_proba(self, X):
        norms = self._compute_residuals(X)
        scale = np.mean(norms) + 1e-6
        d = norms / scale
        exp_neg = np.exp(-d / self.temperature)
        proba = exp_neg / exp_neg.sum(axis=1, keepdims=True)
        return proba

# Registro para serialización
setattr(sys.modules['__main__'], 'SVDSubspaceClassifier', SVDSubspaceClassifier)
print("Clase SVDSubspaceClassifier definida correctamente.")

Clase SVDSubspaceClassifier definida correctamente.


### 2.1. Selección del Rango Óptimo $r$ del Clasificador SVD

Realizamos un barrido del número de componentes singulares $r \in \{5, 10, 15, 20, 30, 50\}$ sobre los vectores planos ($64 \times 64 = 4096$ dimensiones) estandarizados, evaluando el F1 Macro en el conjunto de validación.

In [4]:
scaler_flat = StandardScaler()
X_train_flat_sc = scaler_flat.fit_transform(X_train_flat)
X_val_flat_sc   = scaler_flat.transform(X_val_flat)
X_test_flat_sc  = scaler_flat.transform(X_test_flat)

k_sweep_results = []
for k in [5, 10, 15, 20, 30, 50]:
    svd_temp = SVDSubspaceClassifier(k_components=k, temperature=0.25)
    svd_temp.fit(X_train_flat_sc, y_train)
    pred_val = svd_temp.predict(X_val_flat_sc)
    f1_val = f1_score(y_val, pred_val, average="macro", zero_division=0)
    acc_val = accuracy_score(y_val, pred_val)
    k_sweep_results.append({"r (componentes)": k, "F1 Macro (Val)": round(f1_val, 4), "Accuracy (Val)": round(acc_val, 4)})

df_k_sweep = pd.DataFrame(k_sweep_results)
display(df_k_sweep)

best_k = df_k_sweep.loc[df_k_sweep["F1 Macro (Val)"].idxmax(), "r (componentes)"]
print(f"\nMejor rango SVD: r = {best_k}")

   r (componentes)  F1 Macro (Val)  Accuracy (Val)
0                5          0.4270          0.4251
1               10          0.4995          0.5101
2               15          0.5701          0.5830
3               20          0.5462          0.5668
4               30          0.5883          0.6113
5               50          0.5352          0.5709

Mejor rango SVD: r = 30


In [5]:
# Entrenar SVD final con Train+Val
X_train_flat_final_sc = scaler_flat.fit_transform(X_train_flat_final)
X_test_flat_sc = scaler_flat.transform(X_test_flat)

svd_clf = SVDSubspaceClassifier(k_components=int(best_k), temperature=0.25)
svd_clf.fit(X_train_flat_final_sc, y_train_final)

y_pred_svd  = svd_clf.predict(X_test_flat_sc)
y_proba_svd = svd_clf.predict_proba(X_test_flat_sc)

print("Reporte de Clasificación — SVD Subespacios (Test):")
print(classification_report(y_test, y_pred_svd, target_names=class_names, digits=4))


Reporte de Clasificación — SVD Subespacios (Test):
              precision    recall  f1-score   support

  0 - Normal     0.5714    0.5641    0.5677        78
  1 - Dudosa     0.4554    0.6479    0.5349        71
    2 - Leve     0.4762    0.2857    0.3571        35
3 - Moderada     0.7333    0.6667    0.6984        33
  4 - Severa     0.7895    0.4839    0.6000        31

    accuracy                         0.5524       248
   macro avg     0.6052    0.5296    0.5516       248
weighted avg     0.5736    0.5524    0.5500       248



## 3. Support Vector Machine (SVM) con Kernel Gaussiano RBF

El SVM con kernel RBF mapea los 322 descriptores tabulares a un espacio de Hilbert de dimensión infinita mediante la función de kernel:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|_2^2\right), \quad \gamma = \frac{1}{D \cdot \text{Var}(\mathbf{X})}$$

donde $\gamma$ se calcula automáticamente con la estrategia `'scale'` de scikit-learn.

### Barrido de Regularización $C$

El parámetro $C$ controla el compromiso entre maximizar el margen de separación y minimizar las violaciones de clasificación (*soft margin*). Evaluamos $C \in \{0.1, 1.0, 10.0, 50.0\}$.

In [6]:
scaler_tab = StandardScaler()
X_tr_svm  = scaler_tab.fit_transform(X_train)
X_val_svm = scaler_tab.transform(X_val)

svm_sweep_results = []
for c_val in [0.1, 1.0, 10.0, 50.0]:
    svm_temp = SVC(C=c_val, kernel="rbf", gamma="scale", class_weight="balanced",
                   probability=True, random_state=42)
    svm_temp.fit(X_tr_svm, y_train)
    pred_v = svm_temp.predict(X_val_svm)
    f1_v = f1_score(y_val, pred_v, average="macro", zero_division=0)
    acc_v = accuracy_score(y_val, pred_v)
    svm_sweep_results.append({"C": c_val, "F1 Macro (Val)": round(f1_v, 4), "Accuracy (Val)": round(acc_v, 4)})

df_svm_sweep = pd.DataFrame(svm_sweep_results)
display(df_svm_sweep)

best_svm_c = df_svm_sweep.loc[df_svm_sweep["F1 Macro (Val)"].idxmax(), "C"]
print(f"\nMejor C para SVM RBF: C = {best_svm_c}")

C:\Users\fliaP\miniconda3\envs\ml\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\fliaP\miniconda3\envs\ml\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\fliaP\miniconda3\envs\ml\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\fliaP\miniconda3\envs\ml\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and w

In [7]:
# Entrenar SVM final con Train+Val
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_final)
X_test_scaled  = scaler_final.transform(X_test)

svm_clf = SVC(C=float(best_svm_c), kernel="rbf", gamma="scale", class_weight="balanced",
              probability=True, random_state=42)
svm_clf.fit(X_train_scaled, y_train_final)

y_pred_svm  = svm_clf.predict(X_test_scaled)
y_proba_svm = svm_clf.predict_proba(X_test_scaled)

print("Reporte de Clasificación — SVM RBF (Test):")
print(classification_report(y_test, y_pred_svm, target_names=class_names, digits=4))

C:\Users\fliaP\miniconda3\envs\ml\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
Reporte de Clasificación — SVM RBF (Test):
              precision    recall  f1-score   support

  0 - Normal     0.6915    0.8333    0.7558        78
  1 - Dudosa     0.6438    0.6620    0.6528        71
    2 - Leve     0.5500    0.3143    0.4000        35
3 - Moderada     0.7143    0.7576    0.7353        33
  4 - Severa     0.8846    0.7419    0.8070        31

    accuracy                         0.6895       248
   macro avg     0.6968    0.6618    0.6702       248
weighted avg     0.6851    0.6895    0.6798       248



## 4. XGBoost (Extreme Gradient Boosting)

XGBoost construye un ensamble secuencial de árboles de decisión, donde cada nuevo árbol se entrena para corregir los errores residuales del ensamble anterior mediante *gradient boosting*:

$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(\mathbf{x}_i)$$

donde $\eta = 0.05$ es la tasa de aprendizaje (*learning rate*) y $f_t$ es el árbol añadido en la iteración $t$.

### Configuración (siguiendo el repositorio de la cátedra):
- **`n_estimators`:** Número total de árboles (barrido: $\{50, 100, 150, 200, 300\}$).
- **`max_depth=4`:** Profundidad máxima de cada árbol para controlar el sobreajuste.
- **`subsample=0.8`:** Muestreo estocástico del 80% de las filas por árbol.
- **`colsample_bytree=0.8`:** Muestreo del 80% de las columnas por árbol.


In [8]:
xgb_sweep_results = []
for n_est in [50, 100, 150, 200, 300]:
    xgb_temp = XGBClassifier(
        n_estimators=n_est, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="mlogloss", random_state=42,
    )
    xgb_temp.fit(X_train, y_train)
    pred_v = xgb_temp.predict(X_val)
    f1_v = f1_score(y_val, pred_v, average="macro", zero_division=0)
    acc_v = accuracy_score(y_val, pred_v)
    xgb_sweep_results.append({
        "n_estimators": n_est, "F1 Macro (Val)": round(f1_v, 4), "Accuracy (Val)": round(acc_v, 4)
    })

df_xgb_sweep = pd.DataFrame(xgb_sweep_results)
display(df_xgb_sweep)

best_xgb_n = int(df_xgb_sweep.loc[df_xgb_sweep["F1 Macro (Val)"].idxmax(), "n_estimators"])
print(f"\nMejor n_estimators para XGBoost: {best_xgb_n}")

   n_estimators  F1 Macro (Val)  Accuracy (Val)
0            50          0.5295          0.5425
1           100          0.5651          0.5749
2           150          0.5841          0.5951
3           200          0.5774          0.5992
4           300          0.6025          0.6235

Mejor n_estimators para XGBoost: 300


In [9]:
xgb_clf = XGBClassifier(
    n_estimators=best_xgb_n, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="mlogloss", random_state=42,
)
xgb_clf.fit(X_train_final, y_train_final)

y_pred_xgb  = xgb_clf.predict(X_test)
y_proba_xgb = xgb_clf.predict_proba(X_test)

print("Reporte de Clasificación — XGBoost (Test):")
print(classification_report(y_test, y_pred_xgb, target_names=class_names, digits=4))

Reporte de Clasificación — XGBoost (Test):
              precision    recall  f1-score   support

  0 - Normal     0.7381    0.7949    0.7654        78
  1 - Dudosa     0.6333    0.8028    0.7081        71
    2 - Leve     0.7500    0.3429    0.4706        35
3 - Moderada     0.7273    0.7273    0.7273        33
  4 - Severa     0.9600    0.7742    0.8571        31

    accuracy                         0.7218       248
   macro avg     0.7617    0.6884    0.7057       248
weighted avg     0.7361    0.7218    0.7138       248



## 5. Random Forest

Se compara un ensamble de árboles con distintas cantidades de estimadores y tamaños mínimos de hoja. La combinación con mayor F1 macro en validación se vuelve a entrenar con entrenamiento + validación y se evalúa una sola vez en test.


In [10]:
rf_sweep_results = []
for n_est in [100, 200, 300]:
    for leaf_size in [1, 2]:
        rf_temp = RandomForestClassifier(
            n_estimators=n_est,
            max_features="sqrt",
            min_samples_leaf=leaf_size,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42,
        )
        rf_temp.fit(X_train, y_train)
        pred_val_rf = rf_temp.predict(X_val)
        rf_sweep_results.append({
            "n_estimators": n_est,
            "min_samples_leaf": leaf_size,
            "F1 Macro (Val)": round(f1_score(y_val, pred_val_rf, average="macro", zero_division=0), 4),
            "Accuracy (Val)": round(accuracy_score(y_val, pred_val_rf), 4),
        })

df_rf_sweep = pd.DataFrame(rf_sweep_results)
display(df_rf_sweep.sort_values("F1 Macro (Val)", ascending=False))
rf_best_row = df_rf_sweep.loc[df_rf_sweep["F1 Macro (Val)"].idxmax()]
rf_best_n = int(rf_best_row["n_estimators"])
rf_best_leaf = int(rf_best_row["min_samples_leaf"])
print(f"Mejor Random Forest en validación: n_estimators={rf_best_n}, min_samples_leaf={rf_best_leaf}")

rf_clf = RandomForestClassifier(
    n_estimators=rf_best_n,
    max_features="sqrt",
    min_samples_leaf=rf_best_leaf,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf_clf.fit(X_train_final, y_train_final)
y_pred_rf = rf_clf.predict(X_test)
y_proba_rf = rf_clf.predict_proba(X_test)
print("Reporte de Clasificación — Random Forest (Test):")
print(classification_report(y_test, y_pred_rf, target_names=class_names, digits=4))



   n_estimators  min_samples_leaf  F1 Macro (Val)  Accuracy (Val)
3           200                 2          0.5982          0.6113
5           300                 2          0.5981          0.6073
4           300                 1          0.5753          0.5911
1           100                 2          0.5635          0.5789
2           200                 1          0.5635          0.5789
0           100                 1          0.5379          0.5547
Mejor Random Forest en validación: n_estimators=200, min_samples_leaf=2
Reporte de Clasificación — Random Forest (Test):
              precision    recall  f1-score   support

  0 - Normal     0.6905    0.7436    0.7160        78
  1 - Dudosa     0.5444    0.6901    0.6087        71
    2 - Leve     0.5000    0.1714    0.2553        35
3 - Moderada     0.5526    0.6364    0.5915        33
  4 - Severa     0.8333    0.6452    0.7273        31

    accuracy                         0.6210       248
   macro avg     0.6242    0.5773    

## 6. Carga del Modelo Baseline (Regresión Logística)

Para la comparación equitativa, cargamos el modelo baseline serializado en `03_Baseline.ipynb`.


In [11]:
lr_pipeline = joblib.load(MODELS_DIR / "logistic_regression_baseline.joblib")
y_pred_lr  = lr_pipeline.predict(X_test)
y_proba_lr = lr_pipeline.predict_proba(X_test)

print("Regresión Logística Baseline cargada correctamente.")
print(f"F1 Macro en Test: {f1_score(y_test, y_pred_lr, average='macro', zero_division=0):.4f}")


Regresión Logística Baseline cargada correctamente.
F1 Macro en Test: 0.5676


## 7. Benchmark Comparativo Final (5 Modelos)

Consolidamos las métricas de los 5 modelos evaluados sobre el **mismo conjunto de test independiente** ($N=248$). Esta tabla constituye uno de los entregables principales del **TP4** de la cátedra.



In [12]:
models_dict = {
    "SVD Subespacios (Menor Norma)": (y_pred_svd, y_proba_svd),
    "SVM (Kernel RBF)": (y_pred_svm, y_proba_svm),
    f"XGBoost ({best_xgb_n} Árboles)": (y_pred_xgb, y_proba_xgb),
    f"Random Forest ({rf_best_n} árboles)": (y_pred_rf, y_proba_rf),
    "Regresión Logística L2 (Baseline)": (y_pred_lr, y_proba_lr),
}

metrics_list = []
for name, (preds, probas) in models_dict.items():
    metrics_list.append({
        "Modelo": name,
        "Accuracy (%)": round(accuracy_score(y_test, preds) * 100, 2),
        "Balanced Acc (%)": round(balanced_accuracy_score(y_test, preds) * 100, 2),
        "F1 Macro": round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
        "F1 Weighted": round(f1_score(y_test, preds, average="weighted", zero_division=0), 4),
        "Precision Macro": round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
        "Recall Macro": round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
    })

df_comparison = pd.DataFrame(metrics_list).sort_values("F1 Macro", ascending=False).reset_index(drop=True)
display(df_comparison)

best_model_name = df_comparison.loc[0, "Modelo"]
best_f1 = df_comparison.loc[0, "F1 Macro"]
print(f"\n🏆 Mejor modelo: {best_model_name} (F1 Macro = {best_f1})")



                              Modelo  ...  Recall Macro
0              XGBoost (300 Árboles)  ...        0.6884
1                   SVM (Kernel RBF)  ...        0.6618
2        Random Forest (200 árboles)  ...        0.5773
3  Regresión Logística L2 (Baseline)  ...        0.5903
4      SVD Subespacios (Menor Norma)  ...        0.5296

[5 rows x 7 columns]

🏆 Mejor modelo: XGBoost (300 Árboles) (F1 Macro = 0.7057)


## 8. Matrices de Confusión Comparativas

Visualización de las matrices de confusión de los 5 modelos para inspeccionar los patrones de error específicos de cada arquitectura.


In [13]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=150)
axes = axes.flatten()

for ax, (name, (preds, _)) in zip(axes, models_dict.items()):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_short)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    acc_val = accuracy_score(y_test, preds) * 100
    f1_val = f1_score(y_test, preds, average="macro", zero_division=0)
    short_name = name.split("(")[0].strip()
    ax.set_title(f"{short_name}\nAcc={acc_val:.1f}% | F1={f1_val:.3f}", fontsize=10, fontweight="bold")
    ax.grid(False)

for ax in axes[len(models_dict):]:
    ax.axis("off")
plt.suptitle("Matrices de Confusión en Test (N=248) — Benchmark Comparativo",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()

cm_fig_path = FIGURES_DIR / "modelos_avanzados_confusion_matrices.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada: {cm_fig_path}")
plt.show()



Figura guardada: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\modelos_avanzados_confusion_matrices.png
04_Modelos_Avanzados.ipynb:cell23:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 9. Curvas ROC Multiclase Comparativas (One-vs-Rest)

Caracterizamos el compromiso sensibilidad/especificidad de cada modelo independientemente del umbral de decisión.


In [14]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])
n_classes = 5
colors = ["#2ca02c", "#1f77b4", "#ff7f0e", "#d62728", "#9467bd"]

fig, axes_roc = plt.subplots(2, 3, figsize=(18, 10), dpi=150)
axes_roc = axes_roc.flatten()

for ax, (name, (_, probas)) in zip(axes_roc, models_dict.items()):
    for i, color in zip(range(n_classes), colors):
        fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], probas[:, i])
        auc_i = auc(fpr_i, tpr_i)
        ax.plot(fpr_i, tpr_i, color=color, lw=1.5,
                label=f"G{i} {classes_short[i]} ({auc_i:.2f})")

    # Macro-average
    all_fpr = np.unique(np.concatenate(
        [roc_curve(y_test_bin[:, i], probas[:, i])[0] for i in range(n_classes)]
    ))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], probas[:, i])
        mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)

    ax.plot(all_fpr, mean_tpr, "k:", lw=2, label=f"Macro ({macro_auc:.2f})")
    ax.plot([0, 1], [0, 1], "gray", ls="--", lw=1, alpha=0.5)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    short_name = name.split("(")[0].strip()
    ax.set_title(f"{short_name}\nMacro AUC = {macro_auc:.3f}", fontsize=10, fontweight="bold")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, ls=":", alpha=0.5)

for ax in axes_roc[len(models_dict):]:
    ax.axis("off")
plt.suptitle("Curvas ROC Multiclase (One-vs-Rest) — Benchmark en Test",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()

roc_fig_path = FIGURES_DIR / "modelos_avanzados_roc_curves.png"
plt.savefig(roc_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada: {roc_fig_path}")
plt.show()



Figura guardada: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\modelos_avanzados_roc_curves.png
04_Modelos_Avanzados.ipynb:cell25:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 10. Curvas de Aprendizaje de SVM RBF y XGBoost

Para diagnosticar el régimen sesgo/varianza de los modelos no lineales, calculamos las curvas de aprendizaje con validación cruzada estratificada de 5 pliegues y 8 tamaños de muestra.


In [15]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
train_fracs = np.linspace(0.2, 1.0, 5)

# SVM Learning Curve
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(C=50.0, kernel="rbf", gamma="scale",
                class_weight="balanced", random_state=42))
])
ts_svm, tr_sc_svm, val_sc_svm = learning_curve(
    svm_pipe, X_train, y_train, train_sizes=train_fracs,
    cv=cv_strategy, scoring="f1_macro", n_jobs=1, random_state=42
)

# XGBoost Learning Curve
ts_xgb, tr_sc_xgb, val_sc_xgb = learning_curve(
    XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, eval_metric="mlogloss",
                  random_state=42),
    X_train, y_train, train_sizes=train_fracs,
    cv=cv_strategy, scoring="f1_macro", n_jobs=1, random_state=42
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5), dpi=150)

for ax, ts, tr_sc, val_sc, title in [
    (ax1, ts_svm, tr_sc_svm, val_sc_svm, "SVM RBF (C=50.0)"),
    (ax2, ts_xgb, tr_sc_xgb, val_sc_xgb, "XGBoost (300 Árboles)")
]:
    tr_m, tr_s = np.mean(tr_sc, axis=1), np.std(tr_sc, axis=1)
    val_m, val_s = np.mean(val_sc, axis=1), np.std(val_sc, axis=1)
    ax.plot(ts, tr_m, "o-", color="#1f77b4", lw=2, label="Train F1 Macro")
    ax.fill_between(ts, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="#1f77b4")
    ax.plot(ts, val_m, "s-", color="#ff7f0e", lw=2, label="CV F1 Macro")
    ax.fill_between(ts, val_m - val_s, val_m + val_s, alpha=0.15, color="#ff7f0e")
    ax.set_title(f"Curva de Aprendizaje — {title}", fontsize=11, pad=10)
    ax.set_xlabel("N muestras", fontsize=10)
    ax.set_ylabel("F1-Score Macro", fontsize=10)
    ax.set_ylim([0.0, 1.0])
    ax.legend(loc="lower right", frameon=True)
    ax.grid(True, ls=":", alpha=0.6)

plt.tight_layout()
lc_fig_path = FIGURES_DIR / "modelos_avanzados_learning_curves.png"
plt.savefig(lc_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada: {lc_fig_path}")
plt.show()

print(f"SVM RBF  — Train final: {np.mean(tr_sc_svm, axis=1)[-1]:.4f}, CV final: {np.mean(val_sc_svm, axis=1)[-1]:.4f}")
print(f"XGBoost  — Train final: {np.mean(tr_sc_xgb, axis=1)[-1]:.4f}, CV final: {np.mean(val_sc_xgb, axis=1)[-1]:.4f}")


Figura guardada: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\modelos_avanzados_learning_curves.png
04_Modelos_Avanzados.ipynb:cell27:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
SVM RBF  — Train final: 0.9994, CV final: 0.6089
XGBoost  — Train final: 0.9994, CV final: 0.5733


## 11. Importancia de Características (XGBoost)

Una ventaja clave de los modelos basados en árboles es su capacidad de cuantificar la **importancia de cada descriptor** para la clasificación. Esto permite identificar qué aspectos radiológicos (textura HOG, gradientes Sobel, estadísticos de intensidad) aportan mayor valor discriminante.


In [16]:
importances = xgb_clf.feature_importances_
top_idx = np.argsort(importances)[-20:]

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.barh(range(20), importances[top_idx], color="#2ca02c", alpha=0.8)
ax.set_yticks(range(20))
ax.set_yticklabels([feature_names[i] for i in top_idx], fontsize=9)
ax.set_xlabel("Importancia (Ganancia)", fontsize=11)
ax.set_title(f"Top 20 Características más Importantes — XGBoost ({best_xgb_n} Árboles)",
             fontsize=12, pad=10)
ax.grid(True, axis="x", ls=":", alpha=0.6)
plt.tight_layout()

fi_fig_path = FIGURES_DIR / "modelos_avanzados_feature_importance.png"
plt.savefig(fi_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada: {fi_fig_path}")
plt.show()

print("\nTop 5 características más importantes:")
for i in np.argsort(importances)[-5:][::-1]:
    print(f"  - {feature_names[i]}: {importances[i]:.4f}")


Figura guardada: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\modelos_avanzados_feature_importance.png
04_Modelos_Avanzados.ipynb:cell29:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown

Top 5 características más importantes:
  - grad_mean: 0.0162
  - hog_24: 0.0150
  - hog_63: 0.0131
  - hog_258: 0.0102
  - hog_48: 0.0079


## 12. Serialización de Modelos y Métricas

Exportamos todos los artefactos entrenados y el registro consolidado de métricas para la presentación final (TP4).


In [17]:
# Guardar modelos
joblib.dump(svd_clf, MODELS_DIR / "svd_subspace_classifier.joblib")
joblib.dump(scaler_flat, MODELS_DIR / "scaler_flat.joblib")
joblib.dump(svm_clf, MODELS_DIR / "svm_rbf.joblib")
joblib.dump(scaler_final, MODELS_DIR / "scaler_tabular.joblib")
joblib.dump(xgb_clf, MODELS_DIR / "xgboost_forest.joblib")
joblib.dump(rf_clf, MODELS_DIR / "random_forest.joblib")

# Guardar métricas
all_metrics = {
    "project": "Clasificacion Artrosis de Rodilla - ML1 FIUNA",
    "best_model": best_model_name,
    "best_model_f1_macro": float(best_f1),
    "comparison_table": metrics_list,
    "svd_best_k": int(best_k),
    "svm_best_C": float(best_svm_c),
    "xgb_best_n_estimators": int(best_xgb_n),
    "rf_best_n_estimators": int(rf_best_n),
    "rf_best_min_samples_leaf": int(rf_best_leaf),
}

metrics_json_path = REPORTS_DIR / "modelos_avanzados_metrics.json"
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)

print("Artefactos generados:")
print(f"  1. {MODELS_DIR / 'svd_subspace_classifier.joblib'}")
print(f"  2. {MODELS_DIR / 'svm_rbf.joblib'}")
print(f"  3. {MODELS_DIR / 'xgboost_forest.joblib'}")
print(f"  4. {MODELS_DIR / 'random_forest.joblib'}")
print(f"  4. {MODELS_DIR / 'scaler_tabular.joblib'}")
print(f"  5. {MODELS_DIR / 'scaler_flat.joblib'}")
print(f"  6. {metrics_json_path}")


Artefactos generados:
  1. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\svd_subspace_classifier.joblib
  2. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\svm_rbf.joblib
  3. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\xgboost_forest.joblib
  4. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\random_forest.joblib
  4. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\scaler_tabular.joblib
  5. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\models\scaler_flat.joblib
  6. C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\modelos_avanzados_metrics.json


---
## Conclusiones y análisis comparativo

Se comparan cinco modelos sobre una partición independiente de prueba (N=248): SVD por subespacios, SVM RBF, Random Forest, XGBoost y regresión logística como baseline. La tabla anterior ordena los modelos por F1 macro; la selección de hiperparámetros se realiza con el conjunto de validación.

**Interpretación:** F1 macro da el mismo peso a los cinco grados. La accuracy y balanced accuracy complementan la comparación, mientras que las matrices de confusión muestran qué clases se confunden. La importancia de características corresponde al modelo XGBoost y no implica causalidad.

**Limitaciones:** el conjunto tiene 1.650 imágenes y clases desbalanceadas; la evaluación corresponde a una sola partición. Los resultados no demuestran generalización clínica ni permiten uso diagnóstico.

Los artefactos se guardan en `models/` y `reports/`. El notebook `05_Resultados.ipynb` consolida las métricas para el informe y la página de presentación.
